# Regional Operating Points Efficiency Table Consolidation

This notebook scans holdout test evaluation results across all available kinematic regions ($E_T \in [0, 4], \eta \in [0, 4]$) for a target model configuration, selects the best model per region, and evaluates three operating point methods:
1. **Calibrated Hard Cut (Target PD: 94.24%)**
2. **Pileup-Corrected Cut $T(\mu)$ (Target PD)**
3. **Athena Loose Cut**

The results are formatted into a 2D multi-index grid table grouped by $\eta$ and $E_T$ intervals ($P_D [\%]$, $SP [\%]$, $P_F [\%]$), exported to **Parquet (`.parquet`)**, and compiled into a publication-ready **LaTeX (`.tex`)** table matching the reference paper layout.

In [1]:
from pathlib import Path
import sys

# Resolve repository root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.RegionalOperatingPointsConsolidator import (
    RegionalOperatingPointsConsolidator,
)

print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: /eos/home-i04/p/pmourafr/egamma_master


In [2]:
# Model & Results Paths
CONFIG_PATH = PROJECT_ROOT / "config/NeuralRinger/ModelV9_HighBatch_100Rings.yaml"
RESULTS_PATH = PROJECT_ROOT / "results/config_nameModelV9_HighBatch_100Rings_id20261105222012"
MANIFEST_PATH = RESULTS_PATH.parent / "split_manifest.json"

# Evaluation & Benchmark Configuration
TARGET_PD = 0.9424
WORKING_POINT = "loose"       # Athena fast photon cut WP ('loose', 'medium', 'tight')
EFFICIENCIES_CSV = None       # Optional Path to reference efficiencies CSV
BINS_MU = 10
N_BOOTSTRAPS_PILEUP = 100

# Formatting & Display Options
AS_PERCENTAGE = True          # Scale metrics by 100
PRECISION = 2                 # Decimal precision
DECIMAL_COMMA = False         # Use comma as decimal separator (e.g. 94,24)
MATH_DELIMITERS = True        # Wrap cell text with $...$
BOLD_BEST_SP = True           # Bold the highest SP method within each region

# Export Destinations
OUTPUT_DIR = PROJECT_ROOT / "notebooks/Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_OUTPUT = OUTPUT_DIR / "operating_points_efficiency.parquet"
LATEX_OUTPUT = OUTPUT_DIR / "operating_points_efficiency.tex"

print(f"Config Path:    {CONFIG_PATH}")
print(f"Results Path:   {RESULTS_PATH}")
print(f"Target PD:      {TARGET_PD*100:.2f}%")
print(f"Athena WP:      {WORKING_POINT}")
print(f"Parquet Output: {PARQUET_OUTPUT}")
print(f"LaTeX Output:   {LATEX_OUTPUT}")

Config Path:    /eos/home-i04/p/pmourafr/egamma_master/config/NeuralRinger/ModelV9_HighBatch_100Rings.yaml
Results Path:   /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012
Target PD:      94.24%
Athena WP:      loose
Parquet Output: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_efficiency.parquet
LaTeX Output:   /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_efficiency.tex


In [3]:
consolidator = RegionalOperatingPointsConsolidator(
    config_path=CONFIG_PATH,
    results_path=RESULTS_PATH,
    manifest_path=MANIFEST_PATH if MANIFEST_PATH.exists() else None,
    target_pd=TARGET_PD,
    working_point=WORKING_POINT,
    bins_mu=BINS_MU,
    n_bootstraps_pileup=N_BOOTSTRAPS_PILEUP,
    efficiencies_csv=EFFICIENCIES_CSV,
)

# Run evaluation across all valid kinematic regions
records = consolidator.scan_and_evaluate()
print(f"Total regional method records extracted: {len(records)}")

Skipping unreadable result file /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012/repeat1.fold_idx3.iet4.ieta0.pkl: Ran out of input


Total regional method records extracted: 75


In [4]:
tidy_df = consolidator.build_tidy_dataframe()
display(tidy_df.head(10))

,iet,ieta,method_key,method,target_pd,decision_threshold,pd,pd_uncertainty,sp,sp_uncertainty,...,pf_uncertainty,eff,eff_uncertainty,bg_eff,bg_eff_uncertainty,tp,fp,tn,fn,total
0,0,0,calibrated_hard_cut,Calibrated Hard Cut (Target PD: 94.24%),0.9424,0.2109,0.941529,0.009085,0.489054,0.012003,...,0.013468,0.520792,0.012835,0.189858,0.013468,628,687,161,39,1515
1,0,0,pileup_corrected_cut,Pileup-Corrected Cut T(mu) (Target PD),0.9424,T(mu)=0.0012*mu+-1.3099,0.929535,0.009910,0.501653,0.011963,...,0.013985,0.526733,0.012828,0.209906,0.013985,620,670,178,47,1515
2,0,0,athena_cut,Athena Loose Cut,0.9424,15.0 GeV,0.202399,0.015557,0.489955,0.013273,...,0.009632,0.600660,0.012583,0.913915,0.009632,135,73,775,532,1515
3,0,1,calibrated_hard_cut,Calibrated Hard Cut (Target PD: 94.24%),0.9424,0.2908,0.941909,0.010655,0.541169,0.015236,...,0.019421,0.590863,0.015666,0.254473,0.019421,454,375,128,28,985
4,0,1,pileup_corrected_cut,Pileup-Corrected Cut T(mu) (Target PD),0.9424,T(mu)=-0.0004*mu+-0.8143,0.933610,0.011340,0.547020,0.015216,...,0.019711,0.592893,0.015654,0.266402,0.019711,450,369,134,32,985
5,0,1,athena_cut,Athena Loose Cut,0.9424,15.0 GeV,0.242739,0.019528,0.490299,0.015661,...,0.016787,0.542132,0.015875,0.829026,0.016787,117,86,417,365,985
6,0,2,calibrated_hard_cut,Calibrated Hard Cut (Target PD: 94.24%),0.9424,0.0690,0.938389,0.016553,0.504179,0.013144,...,0.014308,0.360157,0.015038,0.209158,0.014308,198,639,169,13,1019
7,0,2,pileup_corrected_cut,Pileup-Corrected Cut T(mu) (Target PD),0.9424,T(mu)=-0.0066*mu+-2.0649,0.914692,0.019231,0.555819,0.013538,...,0.015937,0.418057,0.015452,0.288366,0.015937,193,575,233,18,1019
8,0,2,athena_cut,Athena Loose Cut,0.9424,15.0 GeV,0.345972,0.032747,0.555944,0.021710,...,0.013608,0.719333,0.014076,0.816832,0.013608,73,148,660,138,1019
9,0,3,calibrated_hard_cut,Calibrated Hard Cut (Target PD: 94.24%),0.9424,0.1198,0.941259,0.008794,0.616698,0.008964,...,0.012441,0.549615,0.010591,0.361930,0.012441,673,952,540,42,2207


In [5]:
grid_table = consolidator.build_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)

display(grid_table)

ET                                                               $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                                                 P_D [%]   
eta                      Method                                                                  
$0.00 \le |\eta| < 0.80$ Calibrated Hard Cut (Target PD: 94.24%)              $94.15 \pm 0.91$   
                         Pileup-Corrected Cut T(mu) (Target PD)      $\mathbf{92.95 \pm 0.99}$   
                         Athena Loose Cut                                     $20.24 \pm 1.56$   
$0.80 \le |\eta| < 1.37$ Calibrated Hard Cut (Target PD: 94.24%)              $94.19 \pm 1.07$   
                         Pileup-Corrected Cut T(mu) (Target PD)      $\mathbf{93.36 \pm 1.13}$   
                         Athena Loose Cut                                     $24.27 \pm 1.95$   
$1.37 \le |\eta| < 1.54$ Calibrated Hard Cut (Target PD: 94.24%)              $93.84 \pm 1.66$   
                         Pileup-Corrected Cut T(mu) (Target PD)               $91.47 \pm 1.92$   
                         Athena Loose Cut                            $\mathbf{34.60 \pm 3.27}$   
$1.54 \le |\eta| < 2.37$ Calibrated Hard Cut (Target PD: 94.24%)     $\mathbf{94.13 \pm 0.88}$   
                         Pileup-Corrected Cut T(mu) (Target PD)               $95.24 \pm 0.80$   
                         Athena Loose Cut                                     $15.52 \pm 1.35$   
$2.37 \le |\eta| < 2.50$ Calibrated Hard Cut (Target PD: 94.24%)     $\mathbf{94.34 \pm 3.17}$   
                         Pileup-Corrected Cut T(mu) (Target PD)               $90.57 \pm 4.02$   
                         Athena Loose Cut                                     $20.75 \pm 5.57$   

ET                                                                                           \
Metric                                                                               SP [%]   
eta                      Method                                                               
$0.00 \le |\eta| < 0.80$ Calibrated Hard Cut (Target PD: 94.24%)           $48.91 \pm 1.20$   
                         Pileup-Corrected Cut T(mu) (Target PD)   $\mathbf{50.17 \pm 1.20}$   
                         Athena Loose Cut                                  $49.00 \pm 1.33$   
$0.80 \le |\eta| < 1.37$ Calibrated Hard Cut (Target PD: 94.24%)           $54.12 \pm 1.52$   
                         Pileup-Corrected Cut T(mu) (Target PD)   $\mathbf{54.70 \pm 1.52}$   
                         Athena Loose Cut                                  $49.03 \pm 1.57$   
$1.37 \le |\eta| < 1.54$ Calibrated Hard Cut (Target PD: 94.24%)           $50.42 \pm 1.31$   
                         Pileup-Corrected Cut T(mu) (Target PD)            $55.58 \pm 1.35$   
                         Athena Loose Cut                         $\mathbf{55.59 \pm 2.17}$   
$1.54 \le |\eta| < 2.37$ Calibrated Hard Cut (Target PD: 94.24%)  $\mathbf{61.67 \pm 0.90}$   
                         Pileup-Corrected Cut T(mu) (Target PD)            $60.63 \pm 0.89$   
                         Athena Loose Cut                                  $43.88 \pm 1.27$   
$2.37 \le |\eta| < 2.50$ Calibrated Hard Cut (Target PD: 94.24%)  $\mathbf{49.83 \pm 5.43}$   
                         Pileup-Corrected Cut T(mu) (Target PD)            $48.51 \pm 5.41$   
                         Athena Loose Cut                                  $49.81 \pm 4.80$   

ET                                                                                           \
Metric                                                                              P_F [%]   
eta                      Method                                                               
$0.00 \le |\eta| < 0.80$ Calibrated Hard Cut (Target PD: 94.24%)           $81.01 \pm 1.35$   
                         Pileup-Corrected Cut T(mu) (Target PD)   $\mathbf{79.01 \pm 1.40}$   
                         Athena Loose Cut        

In [6]:
# Save tidy long-form evaluation data to Parquet
saved_parquet_path = consolidator.export_parquet(PARQUET_OUTPUT)
print(f"Tidy performance table saved to Parquet at: {saved_parquet_path}")

# Also export the MultiIndex grid table to Parquet (flattened columns)
grid_parquet_path = OUTPUT_DIR / "operating_points_efficiency_grid.parquet"
consolidator.export_parquet(grid_parquet_path, df=grid_table)
print(f"Grid table saved to Parquet at: {grid_parquet_path}")

Tidy performance table saved to Parquet at: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_efficiency.parquet
Grid table saved to Parquet at: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_efficiency_grid.parquet


In [7]:
caption = f"Comparative detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions for Calibrated Hard Cut, Pileup-Corrected Cut $T(\mu)$, and Athena {WORKING_POINT.capitalize()} Cut."
label = "tab:operating_points_efficiency_comparison"

latex_code = consolidator.to_latex_snippet(
    grid_table=grid_table,
    caption=caption,
    label=label,
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)

saved_latex_path = consolidator.export_latex(LATEX_OUTPUT, latex_code=latex_code)
print(f"LaTeX table exported successfully to: {saved_latex_path}")
print("\n--- LaTeX Code Snippet ---\n")
print(latex_code)

LaTeX table exported successfully to: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_efficiency.tex

--- LaTeX Code Snippet ---

\begin{table}[htbp]
\centering
\caption{Comparative detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions for Calibrated Hard Cut, Pileup-Corrected Cut $T(\mu)$, and Athena Loose Cut.}
\label{tab:operating_points_efficiency_comparison}
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|l|ccc|ccc|ccc|ccc|ccc|}
\hline
 &  & \multicolumn{3}{c|}{$15 \le E_T < 20\,\text{GeV}$} & \multicolumn{3}{c|}{$20 \le E_T < 30\,\text{GeV}$} & \multicolumn{3}{c|}{$30 \le E_T < 40\,\text{GeV}$} & \multicolumn{3}{c|}{$40 \le E_T < 50\,\text{GeV}$} & \multicolumn{3}{c|}{$E_T \ge 50\,\text{GeV}$} \\
 & Method & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ \\
\hline
\multirow{3}{*}{$0.00 \le |

<>:1: SyntaxWarning: invalid escape sequence '\e'
<>:1: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_1254845/348142016.py:1: SyntaxWarning: invalid escape sequence '\e'
  caption = f"Comparative detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions for Calibrated Hard Cut, Pileup-Corrected Cut $T(\mu)$, and Athena {WORKING_POINT.capitalize()} Cut."


In [8]:
# Training-only table (Best SP fold per region with physical ET and |eta| intervals)
training_records = consolidator.scan_training_best_sp()
print(f"Total regional training best-SP records extracted: {len(training_records)}")

training_grid_table = consolidator.build_training_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=False,
)
display(training_grid_table)

training_latex_code = consolidator.to_latex_snippet(
    grid_table=training_grid_table,
    caption=r"Training best-SP detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions.",
    label="tab:training_best_sp_efficiency",
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)

training_latex_path = consolidator.export_latex(
    OUTPUT_DIR / "operating_points_training_best_sp.tex",
    latex_code=training_latex_code,
)
training_parquet_path = consolidator.export_parquet(
    OUTPUT_DIR / "operating_points_training_best_sp.parquet",
    df=consolidator.build_training_tidy_dataframe(),
)
print(f"Training LaTeX exported to:   {training_latex_path}")
print(f"Training Parquet exported to: {training_parquet_path}")
print("\n--- Training Best-SP LaTeX Code Snippet ---\n")
print(training_latex_code)

Skipping unreadable result file /eos/home-i04/p/pmourafr/egamma_master/results/config_nameModelV9_HighBatch_100Rings_id20261105222012/repeat1.fold_idx3.iet4.ieta0.pkl: Ran out of input


Total regional training best-SP records extracted: 25


ET                                        $15 \le E_T < 20\,\text{GeV}$  \
Metric                                                          P_D [%]   
eta                      Method                                           
$0.00 \le |\eta| < 0.80$ Training Best SP              $64.41 \pm 6.37$   
$0.80 \le |\eta| < 1.37$ Training Best SP              $68.61 \pm 4.53$   
$1.37 \le |\eta| < 1.54$ Training Best SP              $65.71 \pm 4.05$   
$1.54 \le |\eta| < 2.37$ Training Best SP              $74.37 \pm 3.21$   
$2.37 \le |\eta| < 2.50$ Training Best SP              $79.25 \pm 8.60$   

ET                                                                             \
Metric                                               SP [%]           P_F [%]   
eta                      Method                                                 
$0.00 \le |\eta| < 0.80$ Training Best SP  $66.22 \pm 1.12$  $31.96 \pm 6.26$   
$0.80 \le |\eta| < 1.37$ Training Best SP  $71.06 \pm 1.59$  $26.44 \pm 4.08$   
$1.37 \le |\eta| < 1.54$ Training Best SP  $73.01 \pm 2.22$  $19.31 \pm 4.02$   
$1.54 \le |\eta| < 2.37$ Training Best SP  $75.78 \pm 1.19$  $22.79 \pm 2.81$   
$2.37 \le |\eta| < 2.50$ Training Best SP  $77.11 \pm 3.23$  $25.00 \pm 6.78$   

ET                                        $20 \le E_T < 30\,\text{GeV}$  \
Metric                                                          P_D [%]   
eta                      Method                                           
$0.00 \le |\eta| < 0.80$ Training Best SP              $82.26 \pm 0.92$   
$0.80 \le |\eta| < 1.37$ Training Best SP              $81.29 \pm 1.21$   
$1.37 \le |\eta| < 1.54$ Training Best SP              $78.76 \pm 2.08$   
$1.54 \le |\eta| < 2.37$ Training Best SP              $83.25 \pm 0.63$   
$2.37 \le |\eta| < 2.50$ Training Best SP              $84.41 \pm 1.72$   

ET                                                                             \
Metric                                               SP [%]           P_F [%]   
eta                      Method                                                 
$0.00 \le |\eta| < 0.80$ Training Best SP  $86.18 \pm 0.29$   $9.81 \pm 1.02$   
$0.80 \le |\eta| < 1.37$ Training Best SP  $84.76 \pm 0.39$  $11.70 \pm 0.78$   
$1.37 \le |\eta| < 1.54$ Training Best SP  $80.89 \pm 0.59$  $16.96 \pm 1.97$   
$1.54 \le |\eta| < 2.37$ Training Best SP  $85.93 \pm 0.39$  $11.35 \pm 0.62$   
$2.37 \le |\eta| < 2.50$ Training Best SP  $87.07 \pm 0.90$  $10.22 \pm 1.45$   

ET                                        $30 \le E_T < 40\,\text{GeV}$  \
Metric                                                          P_D [%]   
eta                      Method                                           
$0.00 \le |\eta| < 0.80$ Training Best SP              $89.25 \pm 0.51$   
$0.80 \le |\eta| < 1.37$ Training Best SP              $88.83 \pm 0.82$   
$1.37 \le |\eta| < 1.54$ Training Best SP              $81.89 \pm 1.34$   
$1.54 \le |\eta| < 2.37$ Training Best SP              $88.00 \pm 0.49$   
$2.37 \le |\eta| < 2.50$ Training Best SP              $88.80 \pm 0.71$   

ET                                                                             \
Metric                                               SP [%]           P_F [%]   
eta                      Method                                                 
$0.00 \le |\eta| < 0.80$ Training Best SP  $90.21 \pm 0.17$   $8.83 \pm 0.46$   
$0.80 \le |\eta| < 1.37$ Training Best SP  $88.70 \pm 0.27$  $11.44 \pm 0.64$   
$1.37 \le |\eta| < 1.54$ Training Best SP  $84.20 \pm 0.31$  $13.45 \pm 1.81$   
$1.54 \le |\eta| < 2.37$ Training Best SP  $89.10 \pm 0.10$   $9.80 \pm 0.43$   
$2.37 \le |\eta| < 2.50$ Training Best SP  $90.03 \pm 0.45$   $8.73 \pm 0.50$   

ET                                        $40 \le E_T < 50\,\text{GeV}$  \
Metric                                                          P_D [%]   
eta                      Method                                           
$0.00 \le |\eta| < 0.80$ 

Training LaTeX exported to:   /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_training_best_sp.tex
Training Parquet exported to: /eos/home-i04/p/pmourafr/egamma_master/notebooks/Tables/operating_points_training_best_sp.parquet

--- Training Best-SP LaTeX Code Snippet ---

\begin{table}[htbp]
\centering
\caption{Training best-SP detection efficiency ($P_D$), SP index, and fake rate ($P_F$) across $E_T$ and $|\eta|$ regions.}
\label{tab:training_best_sp_efficiency}
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|l|ccc|ccc|ccc|ccc|ccc|}
\hline
 &  & \multicolumn{3}{c|}{$15 \le E_T < 20\,\text{GeV}$} & \multicolumn{3}{c|}{$20 \le E_T < 30\,\text{GeV}$} & \multicolumn{3}{c|}{$30 \le E_T < 40\,\text{GeV}$} & \multicolumn{3}{c|}{$40 \le E_T < 50\,\text{GeV}$} & \multicolumn{3}{c|}{$E_T \ge 50\,\text{GeV}$} \\
 & Method & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ & $SP[\%]$ & $P_F[\%]$ & $P_D[\%]$ 

In [ ]:
# Combined 4-method comparison: Training Best SP + Calibrated Hard Cut (Athena PD) + Pileup-Corrected Cut T(mu) (Athena PD) + Athena Cut
athena_matched_records = consolidator.scan_athena_matched_and_training()
print(f"Total Athena-matched & training records extracted: {len(athena_matched_records)}")

athena_matched_grid_table = consolidator.build_athena_matched_comparison_grid_table(
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
    math_delimiters=MATH_DELIMITERS,
    bold_best_sp=BOLD_BEST_SP,
)
display(athena_matched_grid_table)

athena_matched_latex_code = consolidator.to_latex_snippet(
    grid_table=athena_matched_grid_table,
    caption=rf"Comparison of Training Best SP, Calibrated Hard Cut, and Pileup-Corrected Cut $T(\mu)$ calibrated to full-dataset Athena {WORKING_POINT.capitalize()} $P_D$, alongside Athena {WORKING_POINT.capitalize()} Cut across $E_T$ and $|\eta|$ regions.",
    label="tab:athena_matched_with_training_best_sp",
    resize_box=True,
    as_percentage=AS_PERCENTAGE,
    precision=PRECISION,
    decimal_comma=DECIMAL_COMMA,
)

athena_matched_latex_path = consolidator.export_latex(
    OUTPUT_DIR / "operating_points_athena_matched_with_training.tex",
    latex_code=athena_matched_latex_code,
)
athena_matched_parquet_path = consolidator.export_parquet(
    OUTPUT_DIR / "operating_points_athena_matched_with_training.parquet",
    df=consolidator.build_athena_matched_tidy_dataframe(),
)
print(f"Athena-matched LaTeX exported to:   {athena_matched_latex_path}")
print(f"Athena-matched Parquet exported to: {athena_matched_parquet_path}")
print("\n--- Athena-Matched + Training Best-SP LaTeX Code Snippet ---\n")
print(athena_matched_latex_code)